# GPU V1 — Naive Parallel IoU Matrix Kernel (Non-Maximum Suppression)
CSC14116 — Applied Parallel Programming, Topic A4.

**This notebook requires a CUDA GPU runtime.** On Colab: `Runtime -> Change runtime type -> T4 GPU`. Unlike `cpu_baseline.ipynb`, this will not run on a CPU-only runtime.

Strategy: the GPU computes the full N×N IoU matrix in one kernel launch (one thread per (i, j) pair, embarrassingly parallel since all pairs are independent). The CPU then runs the greedy suppression loop, but with O(1) lookups into the precomputed matrix instead of recomputing IoU each time.

Mirrors `gpu_v1.py`. `load_data` / `run_cpu` are inlined here (copied from `cpu_baseline.ipynb`) since this notebook doesn't have `src/` on its Python path the way the `.py` scripts do when run locally.

In [1]:
import time

import numpy as np

from numba import cuda

if not cuda.is_available():
    raise RuntimeError(
        "No CUDA-capable GPU detected. On Colab: Runtime -> Change runtime type -> GPU."
    )

## load_data / run_cpu
CPU reference (same as `cpu_baseline.py`) — used here to generate synthetic boxes and to verify GPU V1's output.

In [2]:
def load_data(n, seed=0):
    """Generate synthetic (boxes, scores).

    boxes: (n, 4) array of [x1, y1, x2, y2]
    scores: (n,) array of confidence scores in [0, 1)
    """
    rng = np.random.default_rng(seed)
    x1 = rng.uniform(0, 900, size=n)
    y1 = rng.uniform(0, 900, size=n)
    w = rng.uniform(10, 100, size=n)
    h = rng.uniform(10, 100, size=n)
    boxes = np.stack([x1, y1, x1 + w, y1 + h], axis=1).astype(np.float32)
    scores = rng.uniform(0, 1, size=n).astype(np.float32)
    return boxes, scores

In [3]:
def iou_one_to_many(box, boxes):
    """IoU between a single box (4,) and an array of boxes (M, 4)."""
    xx1 = np.maximum(box[0], boxes[:, 0])
    yy1 = np.maximum(box[1], boxes[:, 1])
    xx2 = np.minimum(box[2], boxes[:, 2])
    yy2 = np.minimum(box[3], boxes[:, 3])

    inter_w = np.maximum(0.0, xx2 - xx1)
    inter_h = np.maximum(0.0, yy2 - yy1)
    inter = inter_w * inter_h

    area_box = (box[2] - box[0]) * (box[3] - box[1])
    area_boxes = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])

    union = area_box + area_boxes - inter
    return inter / np.maximum(union, 1e-9)

In [4]:
def run_cpu(boxes, scores, iou_threshold=0.5):
    order = np.argsort(-scores, kind="stable")
    suppressed = np.zeros(len(boxes), dtype=bool)
    keep = []

    for i in range(len(order)):
        idx = order[i]
        if suppressed[idx]:
            continue
        keep.append(idx)

        remaining = order[i + 1:]
        remaining = remaining[~suppressed[remaining]]
        if len(remaining) == 0:
            continue

        ious = iou_one_to_many(boxes[idx], boxes[remaining])
        suppressed[remaining[ious > iou_threshold]] = True

    return np.array(keep, dtype=np.int64)

## IoU matrix CUDA kernel
One thread per (i, j) pair. Grid: `(ceil(N/16), ceil(N/16))` blocks of `16×16` threads — a common sweet spot for 2-D grid kernels.

In [5]:
# 16×16 = 256 threads/block. Each thread block covers a 16×16 tile of the N×N IoU matrix.
_TPB = (16, 16)


@cuda.jit
def _iou_matrix_kernel(boxes, iou_out):
    """Compute iou_out[i, j] = IoU(boxes[i], boxes[j]) for every (i, j) pair.

    boxes   : (N, 4) float32 device array  [x1, y1, x2, y2]
    iou_out : (N, N) float32 device array  (pre-allocated, written by kernel)
    """
    i, j = cuda.grid(2)
    n = boxes.shape[0]

    if i >= n or j >= n:
        return  # out-of-bounds guard for non-square grids

    # intersection top-left / bottom-right
    x1 = max(boxes[i, 0], boxes[j, 0])
    y1 = max(boxes[i, 1], boxes[j, 1])
    x2 = min(boxes[i, 2], boxes[j, 2])
    y2 = min(boxes[i, 3], boxes[j, 3])

    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h

    area_i = (boxes[i, 2] - boxes[i, 0]) * (boxes[i, 3] - boxes[i, 1])
    area_j = (boxes[j, 2] - boxes[j, 0]) * (boxes[j, 3] - boxes[j, 1])
    union = area_i + area_j - inter

    iou_out[i, j] = inter / union if union > 1e-9 else 0.0

In [6]:
def compute_iou_matrix_gpu(boxes: np.ndarray) -> np.ndarray:
    """Upload boxes to the GPU, run the IoU kernel, return the (N, N) matrix on the host."""
    n = boxes.shape[0]
    # contiguous C-layout is required by Numba's CUDA backend
    d_boxes = cuda.to_device(np.ascontiguousarray(boxes, dtype=np.float32))
    d_iou = cuda.device_array((n, n), dtype=np.float32)

    bpg = (
        (n + _TPB[0] - 1) // _TPB[0],
        (n + _TPB[1] - 1) // _TPB[1],
    )
    _iou_matrix_kernel[bpg, _TPB](d_boxes, d_iou)
    cuda.synchronize()

    return d_iou.copy_to_host()

## run_gpu_v1
1. Sort boxes by score (stable, on CPU).
2. Upload score-sorted boxes → GPU; compute N×N IoU matrix with the kernel.
3. Download IoU matrix → CPU.
4. Greedy suppression using *vectorized* NumPy row-slices (no inner Python loop).

In [7]:
def run_gpu_v1(
    boxes: np.ndarray,
    scores: np.ndarray,
    iou_threshold: float = 0.5,
) -> np.ndarray:
    """GPU V1 Non-Maximum Suppression. Returns indices of kept boxes (descending score)."""
    n = len(boxes)
    order = np.argsort(-scores, kind="stable")   # stable: deterministic on score ties

    # Sort boxes/scores into score-descending order before uploading.
    boxes_sorted = np.ascontiguousarray(boxes[order], dtype=np.float32)

    # GPU: compute full N×N IoU matrix (embarrassingly parallel)
    iou_matrix = compute_iou_matrix_gpu(boxes_sorted)

    # CPU: vectorized greedy suppression
    suppressed = np.zeros(n, dtype=bool)
    keep_ranks = []

    for i in range(n):
        if suppressed[i]:
            continue
        keep_ranks.append(i)
        if i + 1 < n:
            # one NumPy broadcast instead of an inner Python loop
            suppressed[i + 1:] |= iou_matrix[i, i + 1:] > iou_threshold

    # Map sorted ranks back to original box indices
    return order[np.array(keep_ranks, dtype=np.int64)]

## benchmark
CPU baseline vs GPU V1, timed side by side. First call triggers Numba JIT compilation, so we warm up on a tiny batch before timing.

In [8]:
def benchmark(ns=(100, 1_000, 10_000), iou_threshold=0.5, seed=0):
    # warm up: JIT compile the kernel on a small problem
    _boxes, _scores = load_data(10, seed=seed)
    _ = run_gpu_v1(_boxes, _scores, iou_threshold)

    header = f"{'N':>8} | {'CPU (s)':>10} | {'GPU V1 (s)':>12} | {'Speedup':>8}"
    print(header)
    print("-" * len(header))

    results = {}
    for n in ns:
        boxes, scores = load_data(n, seed=seed)

        t0 = time.perf_counter()
        run_cpu(boxes, scores, iou_threshold)
        cpu_t = time.perf_counter() - t0

        t0 = time.perf_counter()
        run_gpu_v1(boxes, scores, iou_threshold)
        gpu_t = time.perf_counter() - t0

        speedup = cpu_t / gpu_t
        results[n] = {"cpu": cpu_t, "gpu_v1": gpu_t, "speedup": speedup}
        print(f"{n:>8} | {cpu_t:>10.4f} | {gpu_t:>12.4f} | {speedup:>7.1f}x")

    return results

## Demo run + verify against CPU baseline

In [9]:
boxes, scores = load_data(1000, seed=0)

keep = run_gpu_v1(boxes, scores, iou_threshold=0.5)
print(f"GPU V1 NMS kept {len(keep)}/{len(boxes)} boxes")

cpu_keep = set(run_cpu(boxes, scores, 0.5).tolist())
gpu_keep = set(keep.tolist())
print(f"Exact match with cpu run_cpu: {cpu_keep == gpu_keep}")

GPU V1 NMS kept 923/1000 boxes
Exact match with cpu run_cpu: True


## Benchmark sweep

In [10]:
_ = benchmark()

       N |    CPU (s) |   GPU V1 (s) |  Speedup
-----------------------------------------------
     100 |     0.0069 |       0.0057 |     1.2x


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 49 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


    1000 |     0.1513 |       0.0146 |    10.3x
   10000 |     2.4918 |       0.2557 |     9.7x
